# SSA Antinomy Conflict Analyzer

This notebook backs the mini-app `ssaAntinomyConflictAnalyzer`.

The action cell below runs when the user clicks Run. It may assume
`params_json` is in scope — the runtime injects it from the latest
mini_app_runs row before execution.


In [ ]:
import json, os, sys, importlib
p=json.loads(params_json)
os.makedirs(outdir,exist_ok=True)
appdir=os.path.dirname(outdir)
if appdir not in sys.path: sys.path.insert(0,appdir)
import invariant_guards, governance_store, material_change, governance_tests, material_change_tests, conflict_engine, cl3_suite, invariant_tests
for module in [invariant_guards,governance_store,material_change,governance_tests,material_change_tests,conflict_engine,cl3_suite,invariant_tests]: importlib.reload(module)
import lifecycle_control, controlled_execution, execution_tests, remediation_report
for module in [lifecycle_control,controlled_execution,execution_tests,remediation_report]: importlib.reload(module)
audit_before=invariant_tests.preservation_manifest(appdir)
audit_checks=invariant_tests.run_checks(appdir)
execution_checks=execution_tests.run_checks(appdir)
# Authority-separation review: all prior execution checks plus 14 role/type negatives.
assert len(execution_checks)==37 and all(c['status']=='PASS' for c in execution_checks)
assert governance_store.RECORD_AUTHORITY_ROLES['CONTROLLED_EXECUTION_REQUEST']=='EXECUTION_OPERATOR'
from governance_store import GovernanceStore
# Regression failures are not silently converted into a valid analysis.
os.makedirs(os.path.join(appdir,'records'),exist_ok=True)
contract_checks=governance_tests.run_checks(appdir)
material_checks=material_change_tests.run_checks(appdir)
if not contract_checks or not material_checks or not all(c.get('status')=='PASS' for c in contract_checks+material_checks): raise RuntimeError('Mandatory internal regression checks did not complete.')
result={}
try:
    result=conflict_engine.analyze(p,globals().get('rulesFile'),globals().get('evidenceBundle'))
    subject=(p.get('subjectId') or '').strip()
    if not subject and result.get('synthetic_rules'): subject='fabricated-demo-records-subject'
    if not subject: raise ValueError('Supply an explicit Subject ID before recording a real blocking assessment.')
    result['subject_details']=p.get('subjectDetails','')
    result['run_settings']['subject_input']=p.get('subjectId','')
    result['run_settings']['subject_details_input']=p.get('subjectDetails','')
    store=GovernanceStore(appdir,outdir)
    store.archive_legacy(os.path.join(outdir,'assessment.json'))
    result=store.record_assessment(result,subject)
    store.export(); store.db.close()
except Exception as ex:
    result={'error':str(ex),'automatic_execution_allowed':False,'execution_gate':{'status':'BLOCKED'}}
with open(os.path.join(outdir,'assessment.json'),'w',encoding='utf-8') as f: json.dump(result,f,ensure_ascii=False,indent=2)
for name,checks in [('cr14_checks.json',contract_checks),('cr15_checks.json',material_checks)]:
    with open(os.path.join(outdir,name),'w',encoding='utf-8') as f: json.dump({'scope':'Internal regression checks, not a CL-3 conformity verdict','checks':checks},f,indent=2)
# Mapping also refreshes a separate read-only behavioral conformity report.
conformity=cl3_suite.write_suite(appdir,outdir,mode='local')
assert len(conformity['results'])==26 and all(r['evaluation_result'] in ['PASS','FAIL','NOT_APPLICABLE'] for r in conformity['results'])
assert cl3_suite.verify_evidence_policy(conformity,outdir)
assert all(r['mandatory'] is True and r['applicable'] is True for r in conformity['results'])
assert conformity['overall_result']==('PASS' if all(r['evaluation_result']=='PASS' for r in conformity['results']) else 'FAIL')
audit_checks+=invariant_tests.verify_conformity_boundary(appdir,outdir,conformity)
preservation=invariant_tests.verify_preservation(audit_before,invariant_tests.preservation_manifest(appdir))
assert len(audit_checks)==20 and all(c['status']=='PASS' for c in audit_checks)
assert conformity['evidence_provenance_verified'] is True
import uuid
from governance_store import now
invariant_report={'audit_id':'invariant-audit-'+str(uuid.uuid4()),'created_at':now(),'scope':'Implemented application boundaries only; not institutional authority or full CL-3 certification','invariant':'representation != resolution != decision authority','internal_regression_result':'PASS','checks':audit_checks,'cr14_checks':contract_checks,'cr15_checks':material_checks,'preservation':preservation,'conformity_report_id':conformity['report_id'],'conformity_overall':conformity['overall_result'],'conformity_results':[{'test_id':r['test_id'],'result':r['evaluation_result'],'reason':r['traceable_reason']} for r in conformity['results']],'implementation':conformity['implementation'],'automatic_execution_allowed':False}
with open(os.path.join(outdir,invariant_report['audit_id']+'.json'),'x',encoding='utf-8') as f: json.dump(invariant_report,f,ensure_ascii=False,indent=2)
with open(os.path.join(outdir,'invariant_audit_latest.json'),'w',encoding='utf-8') as f: json.dump(invariant_report,f,ensure_ascii=False,indent=2)
print('Invariant audit: '+str(len(audit_checks))+' checks PASS; all pre-existing rows/files preserved.')
print(conformity['overall_traceable_reason'])
print('Assessment saved; execution blocked. Scoped CL-3 suite: '+conformity['overall_result'])
# Read-only scientific interoperability overlay; existing governance actions and findings are untouched.
if not result.get('error'):
    import semantic_interop
    importlib.reload(semantic_interop)
    semantic_report=semantic_interop.export_registered(appdir,outdir,result['assessment_id'])
    assert semantic_report['roundtrip_preserved'] and len(semantic_report['synthetic_checks'])==11 and all(c['pass'] for c in semantic_report['synthetic_checks'])
    print('Draft local semantic export '+semantic_report['export_id']+'; SHACL 1.0 conforms='+str(semantic_report['conforms'])+'; no governance/precedence/execution grant.')
preservation=invariant_tests.verify_preservation(audit_before,invariant_tests.preservation_manifest(appdir))
remediation=remediation_report.write(appdir,outdir,conformity,execution_checks,contract_checks,material_checks,audit_checks,preservation,semantic_report if not result.get('error') else None)
print('Remediation report '+remediation['report_id']+': '+remediation['scoped_26_test_result']+'; no full CL-3 certification claim.')

In [ ]:
import json, os, sys, importlib
appdir = os.path.dirname(outdir)
if appdir not in sys.path: sys.path.insert(0,appdir)
import governance_store, material_change
importlib.reload(governance_store)
importlib.reload(material_change)
from governance_store import GovernanceStore, strict_loads
p = json.loads(params_json)
report = {}
try:
    store = GovernanceStore(appdir,outdir)
    operation = p.get('governanceOperation','refresh')
    if operation == 'import':
        target = p.get('governanceTarget','')
        if not target: raise ValueError('Select the original blocking assessment first.')
        if p.get('governanceInputMode','paste') == 'file':
            if p.get('useDemoGovernance',True): raise ValueError('The unsigned schema template is illustrative only. Upload your own authorised document; templates never authorise anything.')
            if not globals().get('governanceDocument'): raise ValueError('Select a governance record file.')
            with open(governanceDocument,encoding='utf-8-sig') as f: raw = f.read()
        else: raw = p.get('governanceJson','')
        document = strict_loads(raw)
        if not isinstance(document,dict): raise ValueError('Provide a JSON document object.')
        report = store.ingest(target,document)
    else:
        report = {'message':'History and integrity checks refreshed. Original assessments unchanged. No execution authorised.'}
    store.export()
except Exception as e:
    report = {'error':str(e),'automatic_execution_allowed':False}
with open(os.path.join(outdir,'governance_action.json'),'w',encoding='utf-8') as f: json.dump(report,f,ensure_ascii=False,indent=2)
print(report.get('message') or report.get('error'))

In [ ]:
import json, os, sys, importlib
appdir=os.path.dirname(outdir)
if appdir not in sys.path: sys.path.insert(0,appdir)
import governance_store, material_change
importlib.reload(governance_store)
importlib.reload(material_change)
from governance_store import GovernanceStore
p=json.loads(params_json)
report={}
try:
    store=GovernanceStore(appdir,outdir)
    if p.get('materialChangeOperation','refresh')=='declare':
        report=store.declare_material_change(p.get('governanceTarget',''),p.get('materialChangeCategory','SUBJECT'),p.get('materialChangeDescription',''),p.get('materialChangeEvidence',''))
    else: report={'message':'Version history and re-evaluation/ACTIVE guards refreshed. No execution authorised.'}
    store.export()
except Exception as e: report={'error':str(e),'automatic_execution_allowed':False}
with open(os.path.join(outdir,'material_change_action.json'),'w',encoding='utf-8') as f: json.dump(report,f,ensure_ascii=False,indent=2)
print(report.get('message') or report.get('error'))

In [ ]:
import json, os, sys, importlib
appdir=os.path.dirname(outdir)
if appdir not in sys.path: sys.path.insert(0,appdir)
import governance_store, material_change, conflict_engine, cl3_suite
for module in [governance_store,material_change,conflict_engine,cl3_suite]: importlib.reload(module)
p=json.loads(params_json)
os.makedirs(outdir,exist_ok=True)
if p.get('useDemoCl3',True):
    report=cl3_suite.write_suite(appdir,outdir,mode='local')
else:
    bundle={}
    path=globals().get('cl3Evidence')
    if path:
        try:
            with open(path,encoding='utf-8-sig') as f: bundle=governance_store.strict_loads(f.read())
        except Exception as ex: bundle={'capture_parse_failure':type(ex).__name__+': '+str(ex)}
    else: bundle={'capture_parse_failure':'No captured implementation-evidence file selected.'}
    report=cl3_suite.write_suite(appdir,outdir,bundle=bundle,mode='replay')
print('Scoped CL-3 suite: '+report['overall_result']+'. Conformity assessment grants no execution authority.')

In [ ]:
import os, sys, json, importlib
appdir=os.path.dirname(outdir)
if appdir not in sys.path: sys.path.insert(0,appdir)
import semantic_interop
importlib.reload(semantic_interop)
p=json.loads(params_json)
try:
    report=semantic_interop.export_registered(appdir,outdir,p.get('semanticTarget','__latest__'))
    print('Draft local semantic package: '+report['export_id']+'; SHACL 1.0 conforms='+str(report['conforms'])+'. Governance history unchanged; no authority or execution grant.')
except Exception as ex:
    report={'error':type(ex).__name__+': '+str(ex),'automatic_execution_allowed':False,'governance_authority_granted':False}
    with open(os.path.join(outdir,'semantic_interop_latest.json'),'w',encoding='utf-8') as f: json.dump(report,f,ensure_ascii=False,indent=2)
    print(report['error'])

In [ ]:
import os,sys,json,importlib
appdir=os.path.dirname(outdir)
if appdir not in sys.path: sys.path.insert(0,appdir)
import controlled_execution,governance_store,lifecycle_control,material_change
for module in [lifecycle_control,material_change,controlled_execution]: importlib.reload(module)
p=json.loads(params_json)
s=None
try:
    target=p.get('governanceTarget','')
    if not target: raise ValueError('Select the exact governance case first.')
    request=governance_store.strict_loads(p.get('executionRequestJson',''))
    s=governance_store.GovernanceStore(appdir,outdir)
    event=controlled_execution.execute_explicit(s,target,request)
    s.export()
    report={'event':event,'automatic_execution_allowed':False}
except Exception as ex:
    report={'error':type(ex).__name__+': '+str(ex),'automatic_execution_allowed':False}
finally:
    if s is not None: s.db.close()
with open(os.path.join(outdir,'execution_action.json'),'w',encoding='utf-8') as f: json.dump(report,f,ensure_ascii=False,indent=2)
print(report.get('error') or 'The explicitly requested bounded evidence digest completed; its Execution Event was appended afterwards. No automatic execution authority was created.')